# CS 340 Project Two: Grazioso Salvare Rescue Dashboard

This notebook uses the `AnimalShelter` CRUD module from Project One to retrieve Austin Animal Center records from MongoDB and display an interactive rescue dashboard. Run the cells in order. The dashboard requires access to the course MongoDB server and its `aac.animals` collection.

In [1]:
import os
import getpass
import re

import numpy as np
import pandas as pd
import plotly.express as px
import dash_leaflet as dl
from dash import Dash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output

from animal_shelter import AnimalShelter


In [2]:
# Credentials are requested at runtime and are not stored in the notebook.
username = os.environ.get("AAC_MONGO_USER") or input("MongoDB username: ")
password = os.environ.get("AAC_MONGO_PASSWORD") or getpass.getpass("MongoDB password: ")

shelter = AnimalShelter(username, password)
# A MongoDB empty query is intentional: it retrieves every animal.
records = shelter.read({})
df = pd.DataFrame.from_records(records)
if df.empty:
    raise RuntimeError("No animal records returned. Check your MongoDB connection, credentials, and collection.")

# MongoDB ObjectId instances cannot be serialized to JSON by Dash.
df = df.drop(columns=["_id"], errors="ignore")
df = df.where(pd.notnull(df), None)
print(f"Loaded {len(df):,} animal records.")
print("Available columns:", ", ".join(df.columns))

MongoDB username:  aacuser
MongoDB password:  ········


Loaded 10,004 animal records.
Available columns: rec_num, age_upon_outcome, animal_id, animal_type, breed, color, date_of_birth, datetime, monthyear, name, outcome_subtype, outcome_type, sex_upon_outcome, location_lat, location_long, age_upon_outcome_in_weeks


## Rescue filters

The filters select the recommended animal profiles using breeds, sex, and approximate age in weeks. Breed matching accepts the listed breed names with or without a trailing “Mix” designation. The course's Dashboard Specifications document is the final authority for exact rescue criteria; confirm these lists and age ranges against that document before submission.

In [3]:
# Target animal profiles (age measured in weeks).
RESCUE_PROFILES = {
    "water": {
        "breeds": ["Labrador Retriever", "Chesapeake Bay Retriever", "Newfoundland"],
        "sex": "Intact Female", "min_age": 26, "max_age": 156,
    },
    "mountain": {
        "breeds": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"],
        "sex": "Intact Male", "min_age": 26, "max_age": 156,
    },
    "disaster": {
        "breeds": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"],
        "sex": "Intact Male", "min_age": 20, "max_age": 300,
    },
}


def animal_filter(data, rescue_type):
    """Return a new DataFrame matching the chosen rescue category."""
    if rescue_type == "all":
        return data.copy()
    p = RESCUE_PROFILES[rescue_type]
    required = ("breed", "sex_upon_outcome", "age_upon_outcome_in_weeks")
    missing = [name for name in required if name not in data.columns]
    if missing:
        raise KeyError(f"Missing required animal fields: {missing}")
    breed = data["breed"].fillna("").astype(str)
    pattern = r"(?:" + "|".join(re.escape(b) for b in p["breeds"]) + r")(?: Mix)?"
    breed_match = breed.str.contains(pattern, case=False, regex=True, na=False)
    sex_match = data["sex_upon_outcome"].fillna("").str.casefold() == p["sex"].casefold()
    ages = pd.to_numeric(data["age_upon_outcome_in_weeks"], errors="coerce")
    return data.loc[breed_match & sex_match & ages.between(p["min_age"], p["max_age"])].copy()

# Check how many records match each rescue profile.
for category in ("all", "water", "mountain", "disaster"):
    print(f"{category.title()}: {len(animal_filter(df, category)):,}")

All: 10,004
Water: 25
Mountain: 19
Disaster: 28


In [4]:
# The Model is the shelter read() method and pandas DataFrame.
# The View is the Dash layout, and the Controller is implemented in callbacks.
app = Dash(__name__)

app.layout = html.Div([
     html.Img(
            src=app.get_asset_url("grazioso_logo.png"),
            style={
                "height": "160px",
                "display": "block",
                "margin": "0 auto 15px auto",
                "backgroundColor": "white",
                "padding": "8px",
                "borderRadius": "8px"
            }
        ),
    html.Header([
        html.H1("Grazioso Salvare | Animal Rescue Dashboard", style={"marginBottom": "5px"}),
        html.P("CS 340 Project Two | Justin Beach", style={"marginTop": "0"}),
        html.P("Explore Austin Animal Center animals that meet rescue-training criteria."),
    ], style={"backgroundColor": "#15324A", "color": "white", "padding": "20px", "borderRadius": "10px"}),
    html.Div([
        html.H3("Rescue training category"),
        dcc.RadioItems(id="rescue-filter", options=[
            {"label": "Reset / All Animals", "value": "all"},
            {"label": "Water Rescue", "value": "water"},
            {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
            {"label": "Disaster or Individual Tracking", "value": "disaster"},
        ], value="all", labelStyle={"display": "inline-block", "paddingRight": "22px", "paddingBottom": "10px"}),
        html.Div(id="record-count", style={"fontWeight": "bold"}),
    ], style={"padding": "12px 0"}),
    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": col.replace("_", " ").title(), "id": col, "selectable": True} for col in df.columns],
        data=df.to_dict("records"),
        row_selectable="single", selected_rows=[],
        page_action="native", page_current=0, page_size=10,
        sort_action="native", filter_action="native",
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "minWidth": "115px", "maxWidth": "220px", "whiteSpace": "normal"},
        style_header={"backgroundColor": "#D7E9F1", "fontWeight": "bold"},
        style_data_conditional=[{"if": {"state": "selected"}, "backgroundColor": "#BCE5E1", "border": "1px solid #277C83"}],
    ),
    html.Div([
        html.Div([
            html.H3("Breeds in the filtered table"),
            dcc.Graph(id="breed-chart"),
        ], style={"flex": "1", "minWidth": "340px"}),
        html.Div([
            html.H3("Selected animal location"),
            html.Div(id="map-id"),
        ], style={"flex": "1", "minWidth": "340px"}),
    ], style={"display": "flex", "gap": "20px", "flexWrap": "wrap", "marginTop": "20px"}),
], style={"padding": "18px", "fontFamily": "Arial, sans-serif"})

# Update the table whenever the rescue category changes.
@app.callback(
    [Output("datatable-id", "data"), Output("datatable-id", "page_current"),
     Output("datatable-id", "selected_rows"), Output("record-count", "children")],
    [Input("rescue-filter", "value")],
)
def update_table(rescue_type):
    filtered = animal_filter(df, rescue_type)
    return filtered.to_dict("records"), 0, [], f"Matching animals: {len(filtered):,}"

# A Plotly graph shows the ten most frequent breeds in the currently visible table.
@app.callback(Output("breed-chart", "figure"), [Input("datatable-id", "derived_virtual_data")])
def update_breed_chart(rows):
    current = pd.DataFrame(rows) if rows is not None else df
    if current.empty or "breed" not in current.columns:
        fig = px.bar(title="No matching animal records")
    else:
        counts = current["breed"].fillna("Unknown").value_counts().head(10)
        chart_data = pd.DataFrame({"Breed": counts.index, "Animals": counts.values})
        fig = px.bar(chart_data, x="Animals", y="Breed", orientation="h", title="Top 10 breeds")
        fig.update_layout(yaxis={"categoryorder": "total ascending"})
    fig.update_layout(margin={"l": 15, "r": 15, "t": 50, "b": 20}, height=430)
    return fig

# The map responds to the row selected in the sorted/filtered table.
@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"), Input("datatable-id", "derived_virtual_selected_rows")],
)
def update_map(rows, selected_rows):
    current = pd.DataFrame(rows) if rows is not None else df
    if current.empty:
        return html.P("No animals match these filters.")
    row_number = selected_rows[0] if selected_rows else 0
    if row_number < 0 or row_number >= len(current):
        row_number = 0
    animal = current.iloc[row_number]
    try:
        lat = float(animal["location_lat"])
        lon = float(animal["location_long"])
        if not (np.isfinite(lat) and np.isfinite(lon) and -90 <= lat <= 90 and -180 <= lon <= 180):
            raise ValueError("Coordinates outside valid range")
    except (KeyError, TypeError, ValueError):
        return html.P("This record does not include usable map coordinates. Select another animal.")
    name = str(animal.get("name") or "Unknown")
    breed = str(animal.get("breed") or "Unknown breed")
    return dl.Map(
        center=[lat, lon], zoom=11, style={"width": "100%", "height": "430px"},
        children=[dl.TileLayer(), dl.Marker(position=[lat, lon], children=[
            dl.Tooltip(breed),
            dl.Popup([html.H4(name), html.P(breed)]),
        ])],
    )

print("Dashboard initialized. Start the next cell to view it.")

Dashboard initialized. Start the next cell to view it.


In [ ]:
# In the course Jupyter environment, "external" will expose the dashboard link.
# If port 8050 is in use, change the number to an available port (e.g., 8051).
from IPython.display import display, HTML

dashboard_url = "https://securecraft-almanacelvis-8050.codio.io/"

display(HTML(f'''
<a href="{dashboard_url}" target="_blank"
   style="font-size:18px; font-weight:bold;">
    Open Grazioso Salvare Dashboard
</a>
'''))

app.run(host="0.0.0.0", port=8050, debug=False)

Dash is running on http://0.0.0.0:8050/

 * Serving Flask app '__main__' (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on all addresses.
 * Running on http://10.21.97.143:8050/ (Press CTRL+C to quit)
192.168.10.208 - - [19/Oct/2026 03:37:39] "GET / HTTP/1.1" 200 -
192.168.11.131 - - [19/Oct/2026 03:37:39] "GET /_dash-dependencies HTTP/1.1" 200 -
192.168.11.136 - - [19/Oct/2026 03:37:39] "GET /_dash-layout HTTP/1.1" 200 -
192.168.10.85 - - [19/Oct/2026 03:37:40] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
192.168.10.208 - - [19/Oct/2026 03:37:40] "GET /_dash-component-suites/dash/dcc/async-graph.js HTTP/1.1" 304 -
192.168.10.85 - - [19/Oct/2026 03:37:40] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
192.168.11.131 - - [19/Oct/2026 03:37:40] "GET /assets/grazioso_logo.png HTTP/1.1" 200 -
192.168.10.208 - - [19/Oct/2026 03:37:40] "POST /_dash-update-component HTTP/1.1" 200 -
192.168.10.208 - - [19/Oct/2026 03:37:40] "GET /_dash-component-suites/dash/dcc/async-plotlyjs.js HTTP/1.1" 304 -
192.168.10.85 - - [19/Oct/2026 03:37:41] "P

## Testing checklist

1. Run all cells and verify the MongoDB record count appears.
2. Open the dashboard URL and confirm the unfiltered table is populated.
3. Test all three rescue categories and verify the table and breed chart update.
4. Select an animal row and verify the location map follows that selection.
5. Try native table sorting and filtering, and check that the chart reflects the visible records.
6. Select Reset / All Animals and verify the original count is restored.
7. Capture screenshots for Water Rescue, Mountain/Wilderness, Disaster/Tracking, and Reset, plus any interactive functions required by the rubric.

**Important:** Dataset values, MongoDB server availability, and the exact course-provided breed/age criteria must be verified in the SNHU environment.